In [1]:
import os

# dataset
from lerobot.datasets.transforms import ImageTransformsConfig
from lerobot.datasets.lerobot_dataset import LeRobotDatasetMetadata

# policy
from lerobot.policies.act.configuration_act import ACTConfig

# lerobot
from lerobot.configs.train import TrainPipelineConfig, DatasetConfig
from lerobot.configs.default import WandBConfig
from lerobot.utils.utils import init_logging
from lerobot.configs.types import PolicyFeature, FeatureType

# torch
from torch.cuda import is_available

# wandb
import wandb

# utils
from dotenv import load_dotenv
from pprint import pprint
import sys

# my code
from src.paths import REPO_ROOT, DATASETS_DIR, HF_NAME, POLICIES_DIR
from src.train_extended import train_extended
from src.yolo.yolo_policy_preprocessor import FilterEnvObsProcessorStep

# set up env secrets
load_dotenv(REPO_ROOT / ".env", override=True)

# magic autoreload
%load_ext autoreload
%autoreload 2

/home/jonathan/miniforge3/envs/lerobot-sim-real-sim/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = "cuda" if is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [ ]:
PUSH_TO_HUB       = True
WANDB             = True
REPO_NAME         = 'so101_car_pick_and_place-bbox'
EXPERIMENT_NAME   = 'yolo_v1'
RESUME_CHECKPOINT = None
DATASET_PATH      = DATASETS_DIR / REPO_NAME
DATASET_ID        = f"{HF_NAME}/{REPO_NAME}"
POLICY_ID         = f"{HF_NAME}/act-{REPO_NAME}-{EXPERIMENT_NAME}"

In [4]:
if PUSH_TO_HUB:
    os.system(f"hf auth login --token {os.getenv('HUGGINGFACE_TOKEN')}")
if WANDB:
    wandb.login(key = os.getenv('WANDB_API_KEY'))

In [5]:
ds_meta = LeRobotDatasetMetadata(DATASET_ID, root = DATASET_PATH)
print(f"Total number of episodes: {ds_meta.total_episodes}")
print(f"Average number of frames per episode: {ds_meta.total_frames / ds_meta.total_episodes:.3f}")
print(f"Frames per second used during data collection: {ds_meta.fps}")
print(f"\nRobot type: {ds_meta.robot_type}")
print(f"\nkeys to access images from cameras: {ds_meta.camera_keys}")
print("\nFeatures:")
pprint(ds_meta.features.keys(), width = 40 )

Total number of episodes: 49
Average number of frames per episode: 769.898
Frames per second used during data collection: 30

Robot type: so101_follower

keys to access images from cameras: ['observation.images.wrist_cam', 'observation.images.top_cam']

Features:
dict_keys(['action', 'observation.state', 'observation.images.wrist_cam', 'observation.images.top_cam', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index', 'observation.environment_state'])


In [6]:
# select image transforms
transforms_cfg = ImageTransformsConfig(
    enable             = False, # note about augmentations
    max_num_transforms = 3,
    random_order       = True,
)

# build dataset
dataset_cfg = DatasetConfig(
    repo_id            = DATASET_ID,
    root               = DATASET_PATH.__str__(),   # dataset location
    image_transforms   = transforms_cfg,           # configure image transformations
    use_imagenet_stats = True,                     # normalize image using imagenet weights
    video_backend      = 'torchcodec',
    streaming          = False,
)

### Policy Config
Manually configure policy features - this overrides defaults from the dataset:

In [7]:
input_features = {
    'observation.state': PolicyFeature(
        type  = FeatureType.STATE,
        shape = (6,)
    ),
    'observation.images.wrist_cam': PolicyFeature(
        type  = FeatureType.VISUAL,
        shape = (3,480, 640)
    ),
    'observation.images.top_cam': PolicyFeature(
        type  = FeatureType.VISUAL,
        shape = (3, 480, 640)
    ),
    'observation.environment_state': PolicyFeature(
        type  = FeatureType.ENV,
        shape = (4,)
    )
}
output_features = {
    'action': PolicyFeature(
        type  = FeatureType.ACTION,
        shape = (6,)
    )
}

Additional processors: if needed, specify here:

In [8]:
policy_processors = [
    FilterEnvObsProcessorStep(
        feature_name="observation.environment_state",
        remove_indices=[2, 5]
    )
]

In [9]:
policy_config = ACTConfig(
    n_obs_steps                 = 1,                                  # policy sample freq - should be 1
    chunk_size                  = 100,                                # chunk size outut from the moodel
    n_action_steps              = 100,                                # use only 15 at inference (0.5 sec)
    input_features              = input_features,
    output_features             = output_features,
    device                      = device,
    use_amp                     = False,                              # note about amp
    push_to_hub                 = PUSH_TO_HUB,
    repo_id                     = POLICY_ID,
    vision_backbone             = 'resnet18',
    pretrained_backbone_weights = 'ResNet18_Weights.IMAGENET1K_V1',
    dim_model                   = 512,
    use_vae                     = True,                               # should you use a encoder to learn the style variable
    latent_dim                  = 32,                                 # of the latent Z encoder
    n_vae_encoder_layers        = 4,                                  # of the latent Z encoder
    temporal_ensemble_coeff     = None,                               # temporal averaging coefficient, nominal is 0.01
    kl_weight                   = 1,                                  # defualt is 10
    optimizer_lr                = 1e-5                                # default is 1e-5
    )

WandB logging:

In [10]:
wandb_config = WandBConfig(
    enable           = WANDB,
    disable_artifact = True,
    project          = 'ACT-Real-Sim-Real',
    entity           = 'jonathm126-personal',
    mode             = 'online',
    run_id           = f'act-{REPO_NAME}-{EXPERIMENT_NAME}-train'  # careful - this is a unique ids
)
print(wandb_config.run_id)


act-so101_car_pick_and_place-bbox-yolo_v1-train


Integration:

In [11]:
train_cfg = TrainPipelineConfig(
    dataset                    = dataset_cfg,
    # env                        = env_config,
    policy                     = policy_config,                                                     # policy config
    output_dir                 = POLICIES_DIR / policy_config.type / REPO_NAME / EXPERIMENT_NAME,
    job_name                   = POLICY_ID + '-train',                                              # name
    num_workers                = 4,
    batch_size                 = 12,                                                                 # check for stabillity
    steps                      = 60000,                                                             # check for number of epochs - we want 15-20 epochs
    eval_freq                  = 20000,
    log_freq                   = 200,
    save_checkpoint            = True,
    save_freq                  = 20000,
    use_policy_training_preset = True,                                                              # for scheduler and optimizer
    eval                       = None,
    wandb                      = wandb_config,
)

Train:

In [12]:
if RESUME_CHECKPOINT is None:
    init_logging()
    train_extended(train_cfg, extra_pipeline = policy_processors)
else:
    assert RESUME_CHECKPOINT is not None
    pth = POLICIES_DIR / policy_config.type / REPO_NAME / EXPERIMENT_NAME / 'checkpoints' / resume_checkpoint / 'pretrained_model' / 'train_config.json'
    assert pth.exists()
    sys.argv = [
        "train",
        f"--config_path={pth}",
        "--resume=true",
    ]
    init_logging()
    train_extended.train()

INFO 2025-11-28 01:18:55 extended.py:148 {'batch_size': 12,
 'dataset': {'episodes': None,
             'image_transforms': {'enable': False,
                                  'max_num_transforms': 3,
                                  'random_order': True,
                                  'tfs': {'brightness': {'kwargs': {'brightness': [0.8,
                                                                                   1.2]},
                                                         'type': 'ColorJitter',
                                                         'weight': 1.0},
                                          'contrast': {'kwargs': {'contrast': [0.8,
                                                                               1.2]},
                                                       'type': 'ColorJitter',
                                                       'weight': 1.0},
                                          'hue': {'kwargs': {'hue': [-0.05,
                 

KeyboardInterrupt: 